# Exploratory Data Analysis PySpark — Personal Key Indicators of Heart Disease (BRFSS 2022)
**DS 5110: Big Data Systems · Group 8** — Okenwa Oke, Patrick Opoku, Thomas Davies

**Project question:** *Who is at risk of a heart attack?* Can a classifier find patterns in health, lifestyle and demographic data that are associated with a self-reported history of heart attack (`HadHeartAttack`)?

In [1]:
import os, shutil, glob, warnings
warnings.filterwarnings("ignore", category=FutureWarning)  
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from pyspark.sql import SparkSession, DataFrame, functions as F, types as T
from pyspark.ml.feature import Imputer

warnings.filterwarnings("ignore", category=FutureWarning, module="pyspark")

spark = (SparkSession.builder
         .appName("DS5110-Group8-Cleaning")
         .master("local[6]")                        
         .config("spark.driver.memory", "6g")       
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print(spark.sparkContext.master, spark.sparkContext.defaultParallelism,
      spark.sparkContext.getConf().get("spark.driver.memory", "1g (default)"))

SEED = 42
TARGET = "HadHeartAttack"
DATA_DIR = Path(os.environ.get("HEART_DATA_DIR", "data"))
for sub in ["bronze", "silver", "gold", "reports"]:
    (DATA_DIR / sub).mkdir(parents=True, exist_ok=True)
FIG_DIR = Path("figures/cleaning"); FIG_DIR.mkdir(parents=True, exist_ok=True)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 14:39:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


local[6] 6 6g


In [2]:
BLUE, ORANGE, INK, INK2, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.labelcolor": INK2, "axes.edgecolor": AXIS, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False,
})
def pct_axis(axis, decimals=0):
    axis.set_major_locator(mtick.MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))
    axis.set_major_formatter(mtick.PercentFormatter(1, decimals=decimals))
def save(fig, name): fig.savefig(FIG_DIR / f"{name}.png")

## Data Ingestion

In [3]:
NUMERIC = ["PhysicalHealthDays", "MentalHealthDays", "SleepHours",
           "HeightInMeters", "WeightInKilograms", "BMI"]

COLUMNS = ["State", "Sex", "GeneralHealth", "PhysicalHealthDays", "MentalHealthDays", "LastCheckupTime",
           "PhysicalActivities", "SleepHours", "RemovedTeeth", "HadHeartAttack", "HadAngina", "HadStroke",
           "HadAsthma", "HadSkinCancer", "HadCOPD", "HadDepressiveDisorder", "HadKidneyDisease",
           "HadArthritis", "HadDiabetes", "DeafOrHardOfHearing", "BlindOrVisionDifficulty",
           "DifficultyConcentrating", "DifficultyWalking", "DifficultyDressingBathing", "DifficultyErrands",
           "SmokerStatus", "ECigaretteUsage", "ChestScan", "RaceEthnicityCategory", "AgeCategory",
           "HeightInMeters", "WeightInKilograms", "BMI", "AlcoholDrinkers", "HIVTesting", "FluVaxLast12",
           "PneumoVaxEver", "TetanusLast10Tdap", "HighRiskLastYear", "CovidPos"]

SCHEMA = T.StructType([T.StructField(c, T.DoubleType() if c in NUMERIC else T.StringType(), True)
                       for c in COLUMNS])

CATEGORICAL = [c for c in COLUMNS if c not in NUMERIC and c != TARGET]
FEATURES = [c for c in COLUMNS if c != TARGET]

raw_csv = os.environ.get("RAW_CSV")
if raw_csv is None:
    import kagglehub
    root = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")
    raw_csv = os.path.join(root, "2022", "heart_2022_with_nans.csv")

raw = (spark.read.csv(raw_csv, header=True, schema=SCHEMA, mode="FAILFAST",
                      nullValue="", emptyValue=None, enforceSchema=False))
raw.write.mode("overwrite").parquet(str(DATA_DIR / "bronze" / "heart_2022_raw.parquet"))
bronze = spark.read.parquet(str(DATA_DIR / "bronze" / "heart_2022_raw.parquet")).cache()

N_RAW = bronze.count()
print(f"Bronze: {N_RAW:,} rows × {len(bronze.columns)} columns (from {Path(raw_csv).name})")
bronze.printSchema()

/home/pzf7xb/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bronze: 445,132 rows × 40 columns (from heart_2022_with_nans.csv)
root
 |-- State: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- GeneralHealth: string (nullable = true)
 |-- PhysicalHealthDays: double (nullable = true)
 |-- MentalHealthDays: double (nullable = true)
 |-- LastCheckupTime: string (nullable = true)
 |-- PhysicalActivities: string (nullable = true)
 |-- SleepHours: double (nullable = true)
 |-- RemovedTeeth: string (nullable = true)
 |-- HadHeartAttack: string (nullable = true)
 |-- HadAngina: string (nullable = true)
 |-- HadStroke: string (nullable = true)
 |-- HadAsthma: string (nullable = true)
 |-- HadSkinCancer: string (nullable = true)
 |-- HadCOPD: string (nullable = true)
 |-- HadDepressiveDisorder: string (nullable = true)
 |-- HadKidneyDisease: string (nullable = true)
 |-- HadArthritis: string (nullable = true)
 |-- HadDiabetes: string (nullable = true)
 |-- DeafOrHardOfHearing: string (nullable = true)
 |-- BlindOrVisionDifficulty: string (n

## Missing-data profile
Assessing **how many** missing values, **where**, and **whether they are random** in order to decide how to handle them.

### a. Missingness per column

In [6]:
pip install Jinja2

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [Jinja2]
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pandas.io.formats.style import Styler

y_col = (F.col(TARGET) == "Yes").cast("int")

aggs = []

for c in FEATURES + [TARGET]:
    miss = F.col(c).isNull()
    aggs += [F.sum(miss.cast("int")).alias(f"{c}__n_miss"),
             F.sum((miss & (F.col(TARGET) == "Yes")).cast("int")).alias(f"{c}__pos_miss"),
             F.sum((miss & F.col(TARGET).isNotNull()).cast("int")).alias(f"{c}__lab_miss"),
             F.sum((~miss & (F.col(TARGET) == "Yes")).cast("int")).alias(f"{c}__pos_obs"),
             F.sum((~miss & F.col(TARGET).isNotNull()).cast("int")).alias(f"{c}__lab_obs")]
    
r = bronze.agg(*aggs).first().asDict()

miss = pd.DataFrame([{
    "column": c,
    "n_missing": r[f"{c}__n_miss"], "pct_missing": r[f"{c}__n_miss"] / N_RAW,
    "rate_if_missing": r[f"{c}__pos_miss"] / r[f"{c}__lab_miss"] if r[f"{c}__lab_miss"] else np.nan,
    "rate_if_observed": r[f"{c}__pos_obs"] / r[f"{c}__lab_obs"],
} for c in FEATURES + [TARGET]]).set_index("column").sort_values("pct_missing", ascending=False)
miss["ratio"] = miss.rate_if_missing / miss.rate_if_observed
miss.to_csv(DATA_DIR / "reports" / "missingness_by_column.csv")

print(f"{(miss.n_missing > 0).sum()} of {len(miss)} columns have missing values; "
      f"{int(miss.n_missing.sum()):,} missing cells ({miss.n_missing.sum() / (N_RAW * len(COLUMNS)):.1%} of all cells)")
miss.style.format({"n_missing": "{:,}", "pct_missing": "{:.2%}", "rate_if_missing": "{:.1%}",
                   "rate_if_observed": "{:.1%}", "ratio": "{:.2f}×"}).bar(subset=["pct_missing"], color="#b7d3f6")

38 of 40 columns have missing values; 902,665 missing cells (5.1% of all cells)


,n_missing,pct_missing,rate_if_missing,rate_if_observed,ratio
column,,,,,
TetanusLast10Tdap,"82,516",18.54%,5.4%,5.7%,0.94×
PneumoVaxEver,"77,040",17.31%,4.6%,5.9%,0.77×
HIVTesting,"66,127",14.86%,6.0%,5.6%,1.07×
ChestScan,"56,046",12.59%,5.8%,5.7%,1.02×
CovidPos,"50,764",11.40%,5.5%,5.7%,0.96×
HighRiskLastYear,"50,623",11.37%,5.5%,5.7%,0.97×
BMI,"48,806",10.96%,4.3%,5.8%,0.74×
FluVaxLast12,"47,121",10.59%,5.5%,5.7%,0.96×
AlcoholDrinkers,"46,574",10.46%,5.5%,5.7%,0.97×


In [ ]:
import kagglehub
import pandas as pd
import os

# Download latest version
path = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")

df = pd.read_csv(path + "/2022/heart_2022_no_nans.csv")
df.head(10)

In [3]:
# load the data into DF
import kagglehub
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("ml_classifier").getOrCreate()

path = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")

In [4]:
path

'/home/pzf7xb/.cache/kagglehub/datasets/kamilpytlak/personal-key-indicators-of-heart-disease/versions/6'

In [8]:
# Load training data
filename = path + "/2022/heart_2022_no_nans.csv"
filename

'/home/pzf7xb/.cache/kagglehub/datasets/kamilpytlak/personal-key-indicators-of-heart-disease/versions/6/2022/heart_2022_no_nans.csv'

In [9]:
# read data into dataframe
training = spark.read.csv(filename,  inferSchema=True, header = True)

In [10]:
training.show(2)

26/10/21 15:13:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------+-------------+------------------+----------------+--------------------+------------------+----------+------------+--------------+---------+---------+---------+-------------+-------+---------------------+----------------+------------+-----------+-------------------+-----------------------+-----------------------+-----------------+-------------------------+-----------------+-------------+--------------------+---------+---------------------+------------+--------------+-----------------+-----+---------------+----------+------------+-------------+--------------------+----------------+--------+
|  State|   Sex|GeneralHealth|PhysicalHealthDays|MentalHealthDays|     LastCheckupTime|PhysicalActivities|SleepHours|RemovedTeeth|HadHeartAttack|HadAngina|HadStroke|HadAsthma|HadSkinCancer|HadCOPD|HadDepressiveDisorder|HadKidneyDisease|HadArthritis|HadDiabetes|DeafOrHardOfHearing|BlindOrVisionDifficulty|DifficultyConcentrating|DifficultyWalking|DifficultyDressingBathing|DifficultyErran